# Limpieza - Medicamentos Vitales No Disponibles

In [ ]:
# ============================================================
# 1. LIBRERÍAS
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path
import re
import unicodedata

In [ ]:
# ============================================================
# 2. RUTA DEL ARCHIVO
# ============================================================

archivo = Path("MEDICAMENTOS_VITALES_NO_DISPONIBLES_20260604.csv")

In [ ]:
# ============================================================
# 3. CARGAR DATOS
# ============================================================

df = pd.read_csv(archivo, encoding="utf-8")

print("Filas originales:", df.shape[0])
print("Columnas originales:", df.shape[1])

df.head()

In [ ]:
# ============================================================
# 4. FUNCIONES DE LIMPIEZA
# ============================================================

def normalizar_nombre_columna(col):
    """Convierte nombres de columnas a formato limpio tipo snake_case."""
    col = str(col).strip().lower()

    # Quitar acentos
    col = unicodedata.normalize("NFKD", col)
    col = "".join(c for c in col if not unicodedata.combining(c))

    # Reemplazos
    col = col.replace("/", "_")
    col = col.replace("-", "_")
    col = re.sub(r"[^a-z0-9_ ]", "", col)
    col = col.replace(" ", "_")
    col = re.sub(r"_+", "_", col).strip("_")

    return col


def limpiar_texto(serie):
    """Limpia espacios dobles, saltos de línea y textos vacíos."""
    return (
        serie.astype("string")
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
        .replace({"": pd.NA, "nan": pd.NA, "None": pd.NA})
    )


def limpiar_numero(serie):
    """Convierte números guardados como texto. Acepta coma decimal."""
    s = serie.astype("string").str.strip().str.replace(",", ".", regex=False)
    return pd.to_numeric(s, errors="coerce")


def limpiar_base_medicamentos_vitales(df):
    """Limpia la base de medicamentos vitales no disponibles."""

    df = df.copy()

    # 1. Normalizar nombres originales
    df.columns = [normalizar_nombre_columna(c) for c in df.columns]

    # 2. Renombrar columnas para hacerlas más claras
    renombres = {
        "fecha_de_autorizacion": "fecha_autorizacion",
        "tipo_de_solicitud": "tipo_solicitud",
        "solicitante_importador": "solicitante_importador",
        "principio_activo1": "principio_activo_1",
        "concentracion_delmedicamento1": "concentracion_medicamento_1",
        "unidad_medida1": "unidad_medida_1",
        "principio_activo2": "principio_activo_2",
        "concentracion_del_medicamento2": "concentracion_medicamento_2",
        "unidad_medida2": "unidad_medida_2",
        "forma_farmaceutica": "forma_farmaceutica",
        "nombre_comercial": "nombre_comercial",
        "cantidad_solicitada": "cantidad_solicitada",
        "presentacion_comercial": "presentacion_comercial",
        "diagnostico_cie_1no_reporta": "diagnostico_cie_descripcion",
        "codigo_diagnostico_cie_10": "codigo_diagnostico_cie10"
    }

    df = df.rename(columns=renombres)

    # 3. Limpiar textos
    columnas_texto = df.select_dtypes(include=["object", "string"]).columns

    for col in columnas_texto:
        df[col] = limpiar_texto(df[col])

    # 4. Convertir fecha de autorización
    df["fecha_autorizacion_limpia"] = pd.to_datetime(
        df["fecha_autorizacion"],
        errors="coerce"
    )

    df["anio_autorizacion"] = df["fecha_autorizacion_limpia"].dt.year
    df["mes_autorizacion"] = df["fecha_autorizacion_limpia"].dt.to_period("M").astype("string")

    # 5. Convertir cantidad solicitada
    df["cantidad_solicitada_num"] = limpiar_numero(df["cantidad_solicitada"])

    # 6. Estandarizar unidades frecuentes
    reemplazos_unidades = {
        "mg/ml": "mg/mL",
        "mg/mL": "mg/mL",
        "mcg/ml": "mcg/mL",
        "µg/ml": "mcg/mL",
        "ug/ml": "mcg/mL",
        "µg/mL": "mcg/mL"
    }

    for col in ["unidad_medida_1", "unidad_medida_2"]:
        if col in df.columns:
            df[col] = df[col].replace(reemplazos_unidades)

    return df


df = limpiar_base_medicamentos_vitales(df)

df.head()

In [ ]:
# ============================================================
# 5. DIAGNÓSTICO RÁPIDO
# ============================================================

resumen = pd.DataFrame({
    "columna": df.columns,
    "tipo": df.dtypes.astype(str).values,
    "nulos": df.isna().sum().values,
    "%_nulos": (df.isna().mean().values * 100).round(2),
    "valores_unicos": df.nunique(dropna=True).values
}).sort_values("%_nulos", ascending=False)

print("Filas:", df.shape[0])
print("Columnas:", df.shape[1])
print("IUM únicos:", df["ium"].nunique(dropna=True))
print("Duplicados exactos:", df.duplicated().sum())

resumen

In [ ]:
# ============================================================
# 6. ELIMINAR DUPLICADOS EXACTOS
# ============================================================

duplicados = df[df.duplicated(keep=False)].copy()

print("Registros duplicados exactos:", duplicados.shape[0])

df_limpia = df.drop_duplicates().reset_index(drop=True)

print("Filas antes:", df.shape[0])
print("Filas después:", df_limpia.shape[0])
print("Registros eliminados:", df.shape[0] - df_limpia.shape[0])


In [ ]:
# Ver una muestra de duplicados, si existen
duplicados.head(20)

In [ ]:
# ============================================================
# 7. ALERTAS
# ============================================================

patron_cie10 = r"^[A-Z][0-9]{2}[A-Z0-9]?$"

df_limpia["sin_ium"] = (
    df_limpia["ium"].isna()
    | df_limpia["ium"].eq("VARIOS IUM")
    | df_limpia["ium"].eq("SIN IUM POR SER FITOTERAPEUTICO")
)

df_limpia["sin_diagnostico"] = (
    df_limpia["diagnostico_cie_descripcion"].isna()
    | df_limpia["diagnostico_cie_descripcion"].eq("NO REPORTADO")
    | df_limpia["codigo_diagnostico_cie10"].isna()
    | df_limpia["codigo_diagnostico_cie10"].eq("NO REPORTADO")
)

df_limpia["codigo_cie10_valido"] = (
    df_limpia["codigo_diagnostico_cie10"]
    .astype("string")
    .str.match(patron_cie10)
    .fillna(False)
)

df_limpia["tiene_segundo_principio_activo"] = (
    df_limpia["principio_activo_2"].notna()
    & ~df_limpia["principio_activo_2"].eq("NO APLICA")
)

df_limpia.head()

In [ ]:
# ============================================================
# 8. RESUMEN DF LIMPIO
# ============================================================

resumen_clean = {
    "filas_originales": df.shape[0],
    "filas_limpias": df_limpia.shape[0],
    "duplicados_eliminados": df.shape[0] - df_limpia.shape[0],
    "columnas_limpias": df_limpia.shape[1],
    "ium_unicos": df_limpia["ium"].nunique(dropna=True),
    "solicitantes_importadores_unicos": df_limpia["solicitante_importador"].nunique(dropna=True),
    "tipos_solicitud": df_limpia["tipo_solicitud"].nunique(dropna=True),
    "medicamentos_nombre_comercial_unicos": df_limpia["nombre_comercial"].nunique(dropna=True),
    "registros_sin_ium_o_ium_generico": int(df_limpia["sin_ium"].sum()),
    "registros_sin_diagnostico": int(df_limpia["sin_diagnostico"].sum()),
    "registros_codigo_cie10_invalido": int((~df_limpia["codigo_cie10_valido"]).sum()),
    "registros_con_segundo_principio_activo": int(df_limpia["tiene_segundo_principio_activo"].sum())
}

pd.DataFrame([resumen_clean]).T.rename(columns={0: "valor"})

In [ ]:
# ============================================================
# 9. TABLAS INTERMEDIAS
# ============================================================

base_limpia = df_limpia.copy()

medicamentos = df_limpia[[
    "ium",
    "nombre_comercial",
    "forma_farmaceutica",
    "presentacion_comercial",
    "principio_activo_1",
    "concentracion_medicamento_1",
    "unidad_medida_1",
    "principio_activo_2",
    "concentracion_medicamento_2",
    "unidad_medida_2"
]].drop_duplicates().reset_index(drop=True)

solicitudes = df_limpia[[
    "fecha_autorizacion_limpia",
    "anio_autorizacion",
    "mes_autorizacion",
    "tipo_solicitud",
    "solicitante_importador",
    "ium",
    "cantidad_solicitada_num"
]].drop_duplicates().reset_index(drop=True)

diagnosticos = df_limpia[[
    "ium",
    "diagnostico_cie_descripcion",
    "codigo_diagnostico_cie10",
    "sin_diagnostico",
    "codigo_cie10_valido"
]].drop_duplicates().reset_index(drop=True)

print("Base limpia:", base_limpia.shape)
print("Medicamentos:", medicamentos.shape)
print("Solicitudes:", solicitudes.shape)
print("Diagnósticos:", diagnosticos.shape)

In [ ]:
# ============================================================
# 10. EXPORTAR RESULTADOS
# ============================================================

salida = Path("Tablas_Medicamentos_Vitales")
salida.mkdir(exist_ok=True)

# Archivos CSV
base_limpia.to_csv(salida / "base_limpia.csv", index=False, encoding="utf-8")
medicamentos.to_csv(salida / "medicamentos.csv", index=False, encoding="utf-8")
solicitudes.to_csv(salida / "solicitudes.csv", index=False, encoding="utf-8")
diagnosticos.to_csv(salida / "diagnosticos.csv", index=False, encoding="utf-8")